# 드럼 채보 벤치마크 — GPU 서버 실행 (YourMT3+, S15P31A307-110)

학교 GPU 서버(JupyterHub)에서 YourMT3+를 다른 5개 모델과 같은 곡·같은 점수로 돌린다. 다른 모델은 코랩에서 이미 돌렸으니 여기서는 YourMT3+만 돌리고, 결과를 내려받아 로컬에서 코랩 결과와 합친다 (채보 설계 1-3절·#11). 셀마다 명령만 부르고 로직은 없다.

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py`로 만든 `data/upload/server/`의 zip 5개와 이 노트북을 JupyterHub 홈 폴더에 올린다(파일 목록 화면 오른쪽 위 Upload).

| 파일 | 내용 | 크기 |
| --- | --- | --- |
| `S15P31A307_code.zip` | 저장소 코드 | 약 2MB |
| `data_synth_v1.zip` | 합성곡 5곡 | 약 220MB |
| `data_gmd.zip` | GMD test 연주 45개 | 약 550MB |
| `data_mdb_drums.zip` | MDB Drums 23곡 | 약 350MB |
| `results_separated.zip` | 코랩에서 분리 모델이 뽑아 둔 드럼 32곡 | 약 600MB |

**공용 서버 규칙.** 지정된 GPU 번호만 쓴다. 1번 셀의 `GPU_ID`에 그 번호를 넣으면 이 노트북에서 띄우는 명령이 전부 그 GPU 하나만 본다. 관리자 권한(sudo)이 없어서 전부 홈 폴더 안에 설치한다.

**순서.** 1~6번은 준비(처음 한 번 20~30분, 대부분 설치), 7번은 30초짜리 동작 확인, 8~11번이 본측정, 12번이 결과 묶기다. 위에서부터 차례로 실행한다.

**끊기면.** 브라우저를 닫아도 서버에서 돌던 셀은 끝까지 돈다. 커널이 멈췄거나 다시 시작했으면 1·2·6번 셀을 다시 실행한 뒤(이미 한 설치·압축 풀기는 건너뜀) 끊긴 셀을 그대로 다시 실행한다. runId가 고정돼 있어 끝난 곡은 건너뛰고 남은 곡만 돈다.

In [ ]:
# 1. 설정. GPU_ID에 이 서버에서 지정받은 GPU 번호를 넣음 (공용 서버라 지정된 GPU만 씀)
import os
import sys

GPU_ID = ""  # 예: "3"
assert GPU_ID.strip(), "GPU_ID에 지정받은 GPU 번호를 넣고 다시 실행"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID  # 이 노트북에서 띄우는 모든 명령이 이 GPU 하나만 봄
os.environ["PYTHONUTF8"] = "1"
os.environ["MPLBACKEND"] = "Agg"  # 커널의 그래프 설정이 하네스 환경에 없는 모듈을 가리키지 않게
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
UV = f"{PY} -m uv"
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!df -h $HOME | tail -1
!ls -la $HOME/*.zip

In [ ]:
# 2. 코드·데이터 풀기. 이미 풀려 있으면 건너뜀. 코드 zip을 새로 올렸으면 ~/ssaplash/S15P31A307 을 지우고 다시 실행
if not os.path.isdir(AI):
    !$PY -m zipfile -e $HOME/S15P31A307_code.zip $WORK
for name, folder in [
    ("data_synth_v1", "data/synth_v1"),
    ("data_gmd", "data/gmd"),
    ("data_mdb_drums", "data/mdb_drums/MDB Drums"),  # data/mdb_drums/는 코드 zip의 CHECKSUMS.txt 때문에 이미 있음
    ("results_separated", "data/results/sep-sw-synth"),
]:
    if not os.path.isdir(f"{AI}/{folder}"):
        !$PY -m zipfile -e $HOME/{name}.zip $AI
%cd $AI
!du -sh data/synth_v1 data/gmd data/mdb_drums data/results

In [ ]:
# 3. uv와 하네스 환경. 서버 드라이버가 CUDA 12.8이라 저장소 기본 설정(리눅스는 CUDA 13판 torch)으로는 GPU를 못 잡음.
#    그래서 분리 모델 extra 없이 기본 의존성만 깔고 CUDA 12.8판 torch만 따로 넣음 (하네스는 GPU 확인용으로만 torch를 씀)
#    이후 명령은 uv run 대신 .venv/bin/ssaplash-bench로 불러, uv가 이 torch를 다시 지우지 않게 함
# uv sync가 Python 3.11을 못 받으면(인터넷 막힘): conda create -n py311 python=3.11 -y 뒤 $UV sync --python ~/.conda/envs/py311/bin/python
# pip --user가 막히면: curl -LsSf https://astral.sh/uv/install.sh | sh 로 ~/.local/bin/uv를 깔고 UV = f"{HOME}/.local/bin/uv"
!$PY -m pip install --user -q uv
!$UV --version
!$UV sync
!$UV pip install --python .venv/bin/python torch==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"
# 마지막 줄이 'cuda True NVIDIA H200 NVL'이어야 함. False면 여기서 멈춤 (그대로 가면 CPU로 돌아 시간이 몇십 배 듦)

In [ ]:
# 4. YourMT3+ 별도 환경(.venv-yourmt3, Python 3.10, torch 2.8 CUDA 12.8)과 코드·체크포인트(ext/YourMT3, 약 1.1GB, 판 고정)
#    README '별도 환경 만들기'의 Linux 판. 패키지 판은 로컬(Windows)에서 돌려 본 것과 같음. 이미 있으면 건너뜀
#    transformers는 4.46부터 생성 중 오류가 나서 4.45.1로 고정 (YourMT3 저장소 이슈 15·16)
![ -f .venv-yourmt3/bin/python ] || ($UV venv .venv-yourmt3 --python 3.10 \
    && $UV pip install --python .venv-yourmt3/bin/python torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128 \
    && $UV pip install --python .venv-yourmt3/bin/python "numpy==1.26.4" "transformers==4.45.1" "tokenizers==0.20.3" "huggingface-hub<1.0" \
       "pytorch-lightning==2.5.6" "torchmetrics==1.9.0" "einops==0.8.2" "mido==1.3.3" "mir-eval==0.8.2" "scikit-learn==1.7.2" \
       "scipy==1.15.3" "wandb==0.31.0" "soundfile==0.14.0" "deprecated==1.3.1")
!.venv-yourmt3/bin/python -I scripts/transcribers_ext/yourmt3_transcribe.py download --repo-dir ext/YourMT3
!.venv-yourmt3/bin/python -c "import torch, transformers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'transformers', transformers.__version__)"

In [ ]:
# 5. 데이터 점검. 기대값: synth_v1 5곡, gmd_test 45곡·52.4분, mdb_drums 23곡·21.8분, mdb_drums_unseen 11곡, 누락 0
!.venv/bin/ssaplash-bench check-data configs/bench/datasets/synth_v1.yaml configs/bench/datasets/gmd_test.yaml configs/bench/datasets/mdb_drums.yaml configs/bench/datasets/mdb_drums_unseen.yaml
# 코랩에서 분리 모델이 뽑아 둔 드럼. 32개(합성곡 5곡·MDB 11곡 × 분리 모델 2개)여야 함
!find data/results/sep-*/separated -name drums.wav | wc -l

In [ ]:
# 6. 명령 줄임말. 커널을 다시 시작했으면 1·2번 셀 다음에 이 셀을 다시 실행
M = "configs/bench/transcribers/external/yourmt3.yaml"
DS = "configs/bench/datasets"
SEP = "data/results"  # 코랩 분리 드럼이 들어 있는 run 폴더들의 위치
RUN = f".venv/bin/ssaplash-bench run-transcription --models {M} --out data/results --mlflow-uri none"

In [ ]:
# 7. 동작 확인: 합성곡 5곡 앞 30초. 첫 줄의 '장치 cuda NVIDIA H200 NVL', 마지막의 '실패 0개'를 확인
!$RUN --dataset $DS/synth_v1.yaml --clip 0 30 --run-id ymt3-clip30
!grep "^| yourmt3" data/results/ymt3-clip30/summary.md

## 본측정

runId가 고정돼 있어 끊기면 같은 셀을 다시 실행하면 이어서 돈다. 별도 환경 스크립트는 곡 목록을 통째로 받아 돌기 때문에 run 하나가 끝날 때까지 곡별 줄이 안 보인다. gmd_test(음원 52분)가 가장 길다. `--save-midi N`은 예측 타격을 N곡 MIDI로 남긴다(코랩과 같게).

In [ ]:
# 8. 정답 드럼 트랙 입력 4묶음
!$RUN --dataset $DS/synth_v1.yaml --run-id ymt3-drums-synth --save-midi 5
!$RUN --dataset $DS/gmd_test.yaml --run-id ymt3-drums-gmd --save-midi 3
!$RUN --dataset $DS/mdb_drums.yaml --run-id ymt3-drums-mdb --save-midi 3
!$RUN --dataset $DS/mdb_drums_unseen.yaml --run-id ymt3-drums-mdbu

In [ ]:
# 9. 원곡 입력 (GMD는 반주가 없어 합성곡·MDB 11곡만)
!$RUN --dataset $DS/synth_v1.yaml --input mixture --run-id ymt3-mix-synth
!$RUN --dataset $DS/mdb_drums_unseen.yaml --input mixture --run-id ymt3-mix-mdbu

In [ ]:
# 10. 분리 모델이 뽑은 드럼 입력. 코랩에서 뽑아 둔 wav를 그대로 씀(분리 모델 설치 없음). 마지막 폴더 이름이 결과의 분리 모델 이름
!$RUN --dataset $DS/synth_v1.yaml --input separated --separated-dir $SEP/sep-sw-synth/separated/bs_roformer_sw --run-id ymt3-sep-sw-synth
!$RUN --dataset $DS/mdb_drums_unseen.yaml --input separated --separated-dir $SEP/sep-sw-mdbu/separated/bs_roformer_sw --run-id ymt3-sep-sw-mdbu
!$RUN --dataset $DS/synth_v1.yaml --input separated --separated-dir $SEP/sep-musdb-synth/separated/bs_roformer_musdb --run-id ymt3-sep-musdb-synth
!$RUN --dataset $DS/mdb_drums_unseen.yaml --input separated --separated-dir $SEP/sep-musdb-mdbu/separated/bs_roformer_musdb --run-id ymt3-sep-musdb-mdbu

In [ ]:
# 11. 보조: 배포된 체크포인트 중 공개 드럼 점수 1위인 YPTF+Multi (PS). 비교표 본문에는 앱 기본값만 넣음 (설계 U19)
!.venv/bin/ssaplash-bench run-transcription --models configs/bench/transcribers/external/yourmt3_ps.yaml --out data/results --mlflow-uri none --dataset $DS/gmd_test.yaml --run-id ymt3ps-drums-gmd

In [ ]:
# 12. 결과 확인과 묶기. run마다 '곡/실패' 칸의 실패가 0인지 본 뒤, 입력 wav 사본(inputs/, 약 2GB)은 지우고 나머지를 zip 하나로
!grep -H "^| yourmt3" data/results/ymt3*/summary.md
!rm -rf data/results/ymt3*/inputs
!cd data/results && $PY -m zipfile -c $HOME/results_ymt3.zip ymt3-*/ ymt3ps-*/
!ls -la $HOME/results_ymt3.zip

## 끝나면

홈 폴더의 `results_ymt3.zip`을 파일 목록에서 내려받는다(체크 → Download). 로컬 `AI/data/results/`에 풀면 run 폴더 `ymt3-*`가 코랩 run 폴더 옆에 놓이고, 아래 명령으로 코랩 결과와 합친 보고서를 만든다. 한글 그래프 글꼴 때문에 보고서는 로컬(Windows)에서 만든다. 서버 GPU(H200)와 코랩 GPU(A100)가 섞이므로 보고서 비교표에 GPU 열이 붙는다. 점수는 그대로 비교하고, 속도·GPU 메모리는 GPU가 달라 따로 본다.

```bash
uv run ssaplash-bench report-transcription data/results/{drums-synth,drums-gmd,drums-mdb,drums-mdbu,mix-synth,mix-mdbu,sep-sw-synth,sep-sw-mdbu,sep-musdb-synth,sep-musdb-mdbu}/results.csv data/results/ymt3-{drums-synth,drums-gmd,drums-mdb,drums-mdbu,mix-synth,mix-mdbu,sep-sw-synth,sep-sw-mdbu,sep-musdb-synth,sep-musdb-mdbu}/results.csv --out data/results/combined-ymt3
```